# Phase 3 (Rhetorical Roles) - fine-tuning

Each run loads a Phase 1 checkpoint, reads sentences in overlapping document windows (`context="doc"`) or one at a time (`"sentence"`), mean-pools each sentence and classifies it with a linear layer or a CRF over the sentence sequence. It keeps the best epoch by macro-F1 on CL+IT dev and saves dev/test scores for offline decoding.

**Inputs:** the secret `HF` and one Kaggle dataset per Phase 1 model (weights, config, `bpe_tokenizer.json`), paths in `MODEL_DIRS`. **Output:** `/kaggle/working/phase3_results_bundle.zip` (logs, metrics, submissions, dev/test logits).

`RUNS` lists every experiment of the phase; `RUN_ONLY` selects the runs behind the best submission (`None` = all). Hyperparameters are starting values, not tuned.

In [ ]:
# ============================ CONFIG ============================
MODEL_DIRS = {          # folder (or any file inside it) of each Phase 1 checkpoint; None = auto-discover
    "m2": None,         # Model 2: A-3 decoder, RoPE + GQA          (a3_config.json + a3_model.pt)
    "m3": None,         # Model 3: A-2 GRU encoder-decoder           (gru_config.json + gru_model.pt, or Trainer *.safetensors)
    "m4": "/kaggle/input/datasets/sanskaars22/model-5",   # Model 4: modern decoder (this upload is legal-modern-dec-101m)
    "m5": "/kaggle/input/datasets/sanskaars22/model-4-prefix-lm-labelled-as-m5-in-notebook",   # Model 5: prefix-LM (legal-prefix-lm-101m)
}
DISCOVER_KEYWORDS = {"m2": ["model-2", "model_2", "model2", "dec-rope"], "m3": ["model-3", "model_3", "model3", "gru"],
                     "m4": ["model-4", "model_4", "model4"], "m5": ["model-5", "model_5", "model5", "prefix"]}

COMMON = dict(
    context="doc",                 # "doc": windows of consecutive sentences | "sentence": one sentence per input
    window=1024,                   # context="doc": max tokens per window (<= the model's max_seq_len)
    overlap=True,                  # context="doc": each window starts at the middle of the previous one
    max_sent_tokens=256,           # a longer sentence is cut to its first N tokens (its label is still predicted)
    batch_tokens=4096,             # padded tokens per batch: ~4 windows of 1024, or ~100 single sentences
    epochs=6, grad_accum=2,
    lr_backbone=3e-5,              # pretrained weights (full / partial / kd)
    lr_adapter=3e-4,               # LoRA / rsLoRA / LoRMA / BitFit parameters
    lr_head=1e-3,                  # the new classification head (+ CRF)
    weight_decay=0.01, warmup_frac=0.1, grad_clip=1.0,
    rank=8, alpha=16, adapter_targets="attn+mlp",   # LoRA / rsLoRA / LoRMA
    partial_trainable_frac=0.5,    # partial: train the top half of the layers (+ head)
    head_dropout=0.1,
    head="linear",                 # "linear" (softmax per sentence) or "crf" (linear-chain CRF over the sentence sequence)
    class_weighting="none",        # "none" or "sqrt_inv" (inverse-sqrt label frequency, capped at 10; linear head only)
    attn="bidir",                  # decoders: "bidir" (no causal mask during fine-tuning) or "causal"
    train_on_val=False,            # True: also train on the dev docs and keep the LAST epoch (no model selection possible)
    kd_temperature=2.0, kd_alpha=0.5,
    student_layers=None,           # keep only the lowest N layers of the checkpoint (any method; None = all layers)
    patience=None,                 # stop after this many epochs without dev improvement (None = off)
    save_weights=True,             # save the trainable parameters of the best epoch
    max_train_windows=None,        # debug: cap training windows per epoch
    seed=42,
)

# LoRA / rsLoRA / LoRMA on every model with Linear layers (m2, m4, m5), full fine-tuning on m4 and m5.
# m3 (GRU) has no Linear layers, so the three adapters do not apply to it: set its method to "full", "partial" or "bitfit".
# A KD run is dict(name=..., model=..., method="kd", teacher=<an earlier run's name>, student_layers=N), after its teacher;
# a teacher trained in an earlier session is found as <teacher>/teacher_logits.pt in an input (that session's bundle).
RUNS = [
    dict(name="m2_lora",   model="m2", method="lora"),
    dict(name="m2_rslora", model="m2", method="rslora"),
    dict(name="m2_lorma",  model="m2", method="lorma"),
    dict(name="m4_lora",   model="m4", method="lora"),
    dict(name="m4_rslora", model="m4", method="rslora"),
    dict(name="m4_lorma",  model="m4", method="lorma"),
    dict(name="m4_full",   model="m4", method="full"),
    dict(name="m5_lora",   model="m5", method="lora"),
    dict(name="m5_rslora", model="m5", method="rslora"),
    dict(name="m5_lorma",  model="m5", method="lorma"),
    dict(name="m5_full",   model="m5", method="full"),
    dict(name="m3",        model="m3", method="full"),
    # KD: 4-layer m5 student, teacher m5_full (its teacher_logits.pt comes from the m5_m3 bundle added as an input)
    dict(name="m5_kd_student", model="m5", method="kd", teacher="m5_full", student_layers=4),
    # CRF head (learned role-to-role transitions) on the m5 runs
    dict(name="m5_lora_crf",   model="m5", method="lora",   head="crf"),
    dict(name="m5_rslora_crf", model="m5", method="rslora", head="crf"),
    dict(name="m5_lorma_crf",  model="m5", method="lorma",  head="crf"),
    dict(name="m5_full_crf",   model="m5", method="full",   head="crf"),
    # final runs: the m5 CRF runs trained on train + dev (90 docs). No dev left for model selection: the LAST epoch is
    # kept, so set `epochs` yourself; their dev scores are on training documents and mean nothing.
    dict(name="m5_lora_crf_all",   model="m5", method="lora",   head="crf", train_on_val=True, epochs=...),
    dict(name="m5_rslora_crf_all", model="m5", method="rslora", head="crf", train_on_val=True, epochs=...),
    dict(name="m5_lorma_crf_all",  model="m5", method="lorma",  head="crf", train_on_val=True, epochs=...),
    dict(name="m5_full_crf_all",   model="m5", method="full",   head="crf", train_on_val=True, epochs=...),
    # the same 4 m5 CRF runs with another seed (identical settings): 4 more members for the CRF ensemble
    dict(name="m5_lora_crf_s1",   model="m5", method="lora",   head="crf", seed=1),
    dict(name="m5_rslora_crf_s1", model="m5", method="rslora", head="crf", seed=1),
    dict(name="m5_lorma_crf_s1",  model="m5", method="lorma",  head="crf", seed=1),
    dict(name="m5_full_crf_s1",   model="m5", method="full",   head="crf", seed=1),
]
# two sessions in parallel (each on GPU T4 x2): session A = train on dev, session B = second seed
RUN_ONLY = ["m5_lora_crf", "m5_rslora_crf", "m5_lorma_crf", "m5_full_crf"]   # runs behind the best submission; None = every run in RUNS
SMOKE = False              # True: every selected run does 1 epoch on 16 windows to check that loading, training,
                           # predicting and zipping work; outputs go to /kaggle/working/smoke/

In [ ]:
import os, re, json, glob, math, time, random, zipfile, hashlib, gc, platform, threading
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("p3_work")
if SMOKE: WORK = os.path.join(WORK, "smoke")
os.makedirs(WORK, exist_ok=True)
LOG_PATH = os.path.join(WORK, "phase3_log.txt")
_log = open(LOG_PATH, "a", encoding="utf-8")

_log_lock = threading.Lock()
def log(*a):
    s = " ".join(str(x) for x in a)
    t = threading.current_thread().name
    if t.startswith("gpu"): s = "\n".join(f"[{t}] {l}" for l in s.split("\n"))   # parallel runs: which GPU worker
    with _log_lock:
        print(s, flush=True); _log.write(s + "\n"); _log.flush()

def section(t):
    log("\n" + "=" * 100); log(t); log("=" * 100)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.bfloat16 if (DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
section(f"Phase 3 fine-tuning - {time.strftime('%Y-%m-%d %H:%M:%S')}")
log(f"torch {torch.__version__} | python {platform.python_version()} | device {DEVICE}"
    + (f" ({torch.cuda.get_device_name(0)}, {torch.cuda.device_count()} GPU)" if DEVICE.type == "cuda" else "")
    + f" | amp {AMP_DTYPE}")
log("COMMON =", json.dumps(COMMON))

In [ ]:
# ---- data: Phase_3_RR from the official dataset ----
def load_phase3_files():
    local = os.environ.get("P3_LOCAL_DATA")                       # local testing only
    if local:
        return {f: os.path.join(r, f) for r, _, fs in os.walk(local) for f in fs}
    from huggingface_hub import HfApi, hf_hub_download
    token, why = os.environ.get("HF_TOKEN"), "HF_TOKEN env var not set"
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF")
        except Exception as e:
            why = f"Kaggle secret 'HF' not readable ({e!r})"
    assert token, (f"no Hugging Face token: {why}. In the Kaggle editor: Add-ons -> Secrets -> add/attach a secret "
                   "named exactly HF (value = your hf_... token) and make sure its checkbox is ON for this notebook.")
    try:
        log(f"HF login ok: {HfApi().whoami(token=token)['name']}")
    except Exception as e:
        raise RuntimeError(f"the HF token was found but is rejected by Hugging Face ({e!r}); "
                           "create a new read token at huggingface.co/settings/tokens") from None
    repo = "Exploration-Lab/CS-781-Capstone"
    files = [f for f in HfApi().list_repo_files(repo, repo_type="dataset", token=token) if f.startswith("Phase_3_RR/")]
    return {os.path.basename(f): hf_hub_download(repo, f, repo_type="dataset", token=token) for f in files}

P3 = load_phase3_files()
read = lambda n: json.load(open(P3[n], encoding="utf-8"))
_vocab = read("label_vocab.json")
LABELS = sorted(_vocab, key=_vocab.get) if isinstance(_vocab, dict) else list(_vocab)   # official order
L2I = {l: i for i, l in enumerate(LABELS)}
N_LABELS = len(LABELS)

DOCS = {}           # doc id -> dict(sents, labels (None for test), domain, split)
for dom in ("CL", "IT"):
    for split in ("train", "dev"):
        for x in read(f"{dom}_{split}.json"):
            assert x["id"] not in DOCS, f"duplicate doc id {x['id']}"
            assert len(x["text"]) == len(x["labels"]) and all(l in L2I for l in x["labels"]), x["id"]
            DOCS[x["id"]] = dict(sents=x["text"], labels=x["labels"], domain=dom, split=split)
    _test = read(f"{dom}_test.json")
    _test = _test.items() if isinstance(_test, dict) else [(x["id"], x["text"]) for x in _test]
    for did, sents in _test:
        if isinstance(sents, dict): sents = sents["text"]
        assert did not in DOCS, f"duplicate doc id {did}"
        DOCS[did] = dict(sents=sents, labels=None, domain=dom, split="test")
TRAIN_IDS = [d for d, x in DOCS.items() if x["split"] == "train"]
VAL_IDS = [d for d, x in DOCS.items() if x["split"] == "dev"]
TEST_IDS = [d for d, x in DOCS.items() if x["split"] == "test"]

section("DATA")
log(f"labels ({N_LABELS}): {LABELS}")
for split in ("train", "dev", "test"):
    for dom in ("CL", "IT"):
        ds = [x for x in DOCS.values() if x["split"] == split and x["domain"] == dom]
        log(f"  {split:5s} {dom}: {len(ds):3d} docs, {sum(len(x['sents']) for x in ds):6d} sentences")
for name, ids in (("train", TRAIN_IDS), ("dev", VAL_IDS)):
    c = Counter(l for d in ids for l in DOCS[d]["labels"])
    log(f"{name:5s} label counts: " + ", ".join(f"{l}={c[l]}" for l in LABELS))
_missing = [l for l in LABELS if not any(l in DOCS[d]["labels"] for d in VAL_IDS)]
log(f"labels with no dev sentence: {_missing} - their dev F1 is 0 whatever the model predicts "
    "(as in evaluate_phase3.py), so the dev macro-F1 cannot reach 1")

In [ ]:
# ---- Phase 1 byte-level BPE (same algorithm and ids as LegalBPETokenizer), with character offsets ----
WORD_RE = re.compile(r"\S+")
_SHIFT = 20
_EOW = (1 << _SHIFT) - 1

class OffsetBPE:
    def __init__(self, path):
        with open(path, encoding="utf-8") as f:
            d = json.load(f)
        self.itos = d["itos"]
        sym2id = {tuple(s): i for i, s in enumerate(self.itos) if isinstance(s, list)}
        to_id = lambda t: _EOW if tuple(t) == (256,) else sym2id[tuple(t)]
        self.ranks = {(to_id(l) << _SHIFT) | to_id(r): sym2id[tuple(l) + tuple(r)] for l, r in d["merges"]}
        self.nbytes = [sum(b != 256 for b in s) if isinstance(s, list) else 0 for s in self.itos]
        self.eow_id = len(self.itos)               # LegalBPETokenizer: a bare end-of-word symbol gets id len(itos)
        self.vocab_size = len(self.itos) + 1
        self.cache = {}

    def tokenize_word(self, word):
        out = self.cache.get(word)
        if out is not None: return out
        w = [b + 4 for b in word.encode("utf-8")] + [_EOW]
        ranks = self.ranks
        while len(w) > 1:
            best = None
            for x, y in zip(w, w[1:]):
                n = ranks.get((x << _SHIFT) | y)
                if n is not None and (best is None or n < best[0]): best = (n, x, y)
            if best is None: break
            n, a, b = best; i, res = 0, []
            while i < len(w):
                if i < len(w) - 1 and w[i] == a and w[i + 1] == b: res.append(n); i += 2
                else: res.append(w[i]); i += 1
            w = res
        self.cache[word] = w
        return w

    def encode(self, text):
        """-> ids, char_start, char_end (np arrays). A bare end-of-word token has an empty span at the word end."""
        ids, cs, ce = [], [], []
        for m in WORD_RE.finditer(text):
            word, base = m.group(), m.start()
            b2c = []
            for ci, ch in enumerate(word): b2c.extend([ci] * len(ch.encode("utf-8")))
            p = 0
            for t in self.tokenize_word(word):
                nb = 0 if t == _EOW else self.nbytes[t]
                ids.append(self.eow_id if t == _EOW else t)
                if nb == 0:
                    cs.append(base + len(word)); ce.append(base + len(word))
                else:
                    cs.append(base + b2c[p]); ce.append(base + b2c[p + nb - 1] + 1); p += nb
        return np.array(ids, np.int64), np.array(cs, np.int64), np.array(ce, np.int64)

# ---- From-scratch attention (no nn.MultiheadAttention / F.scaled_dot_product_attention), as in Phase 1 ----
# out = softmax(q k^T * scale, masked) v, computed in query blocks with the softmax statistics kept per row
# (log-sum-exp), so the full (T, S) score matrix is never stored. The backward pass is written by hand and
# recomputes each block's probabilities from the saved log-sum-exp:
#   P = exp(S - lse),  dV = P^T dO,  dP = dO V^T,  dS = P * (dP - rowsum(dO * O)),  dQ = dS K * scale,  dK = dS^T Q * scale
def _block_scores(q_blk, k, scale, attn_mask, is_causal, i, j, kend, offset):
    s = torch.matmul(q_blk, k[:, :, :kend].transpose(-2, -1)).float() * scale
    if attn_mask is not None:
        s = s.masked_fill(~attn_mask[..., i:j, :kend], float("-inf"))
    if is_causal:
        rows = torch.arange(i, j, device=s.device)[:, None] + offset
        s = s.masked_fill(torch.arange(kend, device=s.device)[None, :] > rows, float("-inf"))
    return s


class BlockwiseAttention(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q, k, v, attn_mask, is_causal, scale, block):
        T, S = q.size(-2), k.size(-2)
        offset = S - T
        out = torch.empty_like(q)
        lse = torch.empty(q.shape[:-1], dtype=torch.float32, device=q.device)
        for i in range(0, T, block):
            j = min(i + block, T)
            kend = min(S, offset + j) if is_causal else S
            s = _block_scores(q[:, :, i:j], k, scale, attn_mask, is_causal, i, j, kend, offset)
            m = s.amax(-1, keepdim=True)
            p = torch.exp(s - m)
            l = p.sum(-1, keepdim=True)
            out[:, :, i:j] = torch.matmul((p / l).to(v.dtype), v[:, :, :kend])
            lse[:, :, i:j] = (m + l.log()).squeeze(-1)
        ctx.save_for_backward(q, k, v, out, lse, attn_mask)
        ctx.is_causal, ctx.scale, ctx.block = is_causal, scale, block
        return out

    @staticmethod
    def backward(ctx, d_out):
        q, k, v, out, lse, attn_mask = ctx.saved_tensors
        is_causal, scale, block = ctx.is_causal, ctx.scale, ctx.block
        T, S = q.size(-2), k.size(-2)
        offset = S - T
        d_out = d_out.contiguous()
        dq = torch.empty_like(q)
        dk = torch.zeros(k.shape, dtype=torch.float32, device=k.device)
        dv = torch.zeros(v.shape, dtype=torch.float32, device=v.device)
        delta = (d_out.float() * out.float()).sum(-1, keepdim=True)
        for i in range(0, T, block):
            j = min(i + block, T)
            kend = min(S, offset + j) if is_causal else S
            q_blk, do_blk = q[:, :, i:j], d_out[:, :, i:j]
            p = torch.exp(_block_scores(q_blk, k, scale, attn_mask, is_causal, i, j, kend, offset) - lse[:, :, i:j, None])
            dv[:, :, :kend] += torch.matmul(p.to(v.dtype).transpose(-2, -1), do_blk).float()
            dp = torch.matmul(do_blk, v[:, :, :kend].transpose(-2, -1)).float()
            ds = (p * (dp - delta[:, :, i:j])).to(q.dtype)
            dq[:, :, i:j] = torch.matmul(ds, k[:, :, :kend]) * scale
            dk[:, :, :kend] += torch.matmul(ds.transpose(-2, -1), q_blk).float() * scale
        return dq, dk.to(k.dtype), dv.to(v.dtype), None, None, None, None


def scratch_attention(q, k, v, attn_mask=None, is_causal=False, scale=None, block=128):
    """q (B, H, T, d), k/v (B, H_kv, S, d) with H_kv dividing H (GQA). attn_mask: bool (True = may attend)."""
    scale = q.size(-1) ** -0.5 if scale is None else scale
    B, H, T, d = q.shape
    n_rep = H // k.size(1)
    if n_rep > 1 and not is_causal:
        q = q.reshape(B, k.size(1), n_rep * T, d)
        if attn_mask is not None:
            attn_mask = attn_mask.repeat(1, 1, n_rep, 1)
        out = BlockwiseAttention.apply(q, k, v, attn_mask, False, scale, block)
        return out.reshape(B, H, T, d)
    if n_rep > 1:
        k, v = k.repeat_interleave(n_rep, dim=1), v.repeat_interleave(n_rep, dim=1)
    return BlockwiseAttention.apply(q, k, v, attn_mask, is_causal, scale, block)


def attend(q, k, v, key_keep, causal, scale=None):
    """causal or bidirectional attention; key_keep (B, T) bool marks real tokens (None = no padding)."""
    if key_keep is None:
        return scratch_attention(q, k, v, is_causal=causal, scale=scale)
    T = q.size(-2)
    allowed = key_keep[:, None, None, :].expand(-1, 1, T, -1)
    if causal:
        allowed = allowed & torch.ones(T, T, dtype=torch.bool, device=q.device).tril()
    allowed = allowed | torch.eye(T, dtype=torch.bool, device=q.device)   # padding rows see themselves (no NaN)
    return scratch_attention(q, k, v, attn_mask=allowed, scale=scale)


def rope_cache(T, d_head, base):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2, dtype=torch.float32) / d_head))
    ang = torch.outer(torch.arange(T, dtype=torch.float32), inv_freq)
    ang = torch.cat([ang, ang], dim=-1)
    return ang.cos(), ang.sin()

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, cos, sin):
    return x * cos + rotate_half(x) * sin

In [ ]:
# ---- Phase 1 backbones as encoders. Parameter names match the Phase 1 checkpoints exactly. ----
class A3Attention(nn.Module):              # Model 2 (A-3 CausalSelfAttention: MHA/GQA, RoPE or sinusoidal)
    def __init__(self, c):
        super().__init__()
        self.n_heads, self.n_kv, self.d_head = c["n_heads"], c["n_kv_heads"], c["d_model"] // c["n_heads"]
        self.scale = c.get("attention_scale") or self.d_head ** -0.5
        self.q_proj = nn.Linear(c["d_model"], self.n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(c["d_model"], self.n_kv * self.d_head, bias=False)
        self.v_proj = nn.Linear(c["d_model"], self.n_kv * self.d_head, bias=False)
        self.out_proj = nn.Linear(self.n_heads * self.d_head, c["d_model"], bias=False)
        self.use_rope = c.get("pos_emb_type") == "rope"
        if self.use_rope:
            cos, sin = rope_cache(c["max_seq_len"], self.d_head, c.get("rope_base", 10000.0))
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, key_keep, causal):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv, self.d_head).transpose(1, 2)
        if self.use_rope:
            cos, sin = self.rope_cos[:T][None, None].to(q.dtype), self.rope_sin[:T][None, None].to(q.dtype)
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        out = attend(q, k, v, key_keep, causal, self.scale)
        return self.out_proj(out.transpose(1, 2).reshape(B, T, -1))


class A3Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.norm_first = c.get("norm_first", False)
        self.ln1 = nn.LayerNorm(c["d_model"])
        self.attn = A3Attention(c)
        self.ln2 = nn.LayerNorm(c["d_model"])
        self.ffn = nn.Module()
        self.ffn.net = nn.Sequential(nn.Linear(c["d_model"], c["d_ff"]), nn.GELU(), nn.Linear(c["d_ff"], c["d_model"]), nn.Dropout(0.0))

    def forward(self, x, key_keep, causal):
        if self.norm_first:
            x = x + self.attn(self.ln1(x), key_keep, causal)
            return x + self.ffn.net(self.ln2(x))
        x = self.ln1(x + self.attn(x, key_keep, causal))
        return self.ln2(x + self.ffn.net(x))


class A3Backbone(nn.Module):
    kind = "a3"
    def __init__(self, c):
        super().__init__()
        assert c.get("attention_type", "standard") == "standard", "only the standard (MHA/GQA) A-3 attention is supported"
        self.c, self.hidden_size = c, c["d_model"]
        self.tok_emb = nn.Embedding(c["vocab_size"], c["d_model"])
        if c.get("pos_emb_type") == "sinusoidal":
            pos = torch.arange(c["max_seq_len"], dtype=torch.float32)[:, None]
            div = torch.exp(torch.arange(0, c["d_model"], 2, dtype=torch.float32) * (-math.log(10000.0) / c["d_model"]))
            pe = torch.zeros(c["max_seq_len"], c["d_model"]); pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
            self.register_buffer("pos_emb", pe, persistent=False)
        self.blocks = nn.ModuleList([A3Block(c) for _ in range(c["n_layers"])])
        self.ln_f = nn.LayerNorm(c["d_model"])

    def forward(self, ids, key_keep=None, causal=False):
        x = self.tok_emb(ids)
        if hasattr(self, "pos_emb"): x = x + self.pos_emb[:ids.size(1)][None]
        for b in self.blocks: x = b(x, key_keep, causal)
        return self.ln_f(x)


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps, self.weight = eps, nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)).to(x.dtype) * self.weight


class ModernAttention(A3Attention):         # Models 4/5 GQAttention (always RoPE), same projections
    def __init__(self, c):
        super().__init__({**c, "pos_emb_type": "rope", "attention_scale": None})


class ModernBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.attn_norm = RMSNorm(c["d_model"], c.get("norm_eps", 1e-5))
        self.attn = ModernAttention(c)
        self.mlp_norm = RMSNorm(c["d_model"], c.get("norm_eps", 1e-5))
        self.mlp = nn.Module()
        self.mlp.gate = nn.Linear(c["d_model"], c["d_ff"], bias=False)
        self.mlp.up = nn.Linear(c["d_model"], c["d_ff"], bias=False)
        self.mlp.down = nn.Linear(c["d_ff"], c["d_model"], bias=False)

    def forward(self, x, key_keep, causal):
        x = x + self.attn(self.attn_norm(x), key_keep, causal)
        h = self.mlp_norm(x)
        return x + self.mlp.down(F.silu(self.mlp.gate(h)) * self.mlp.up(h))


class ModernBackbone(nn.Module):
    kind = "modern"
    def __init__(self, c):
        super().__init__()
        self.c, self.hidden_size = c, c["d_model"]
        self.tok_emb = nn.Embedding(c["vocab_size"], c["d_model"])
        self.blocks = nn.ModuleList([ModernBlock(c) for _ in range(c["n_layers"])])
        self.final_norm = RMSNorm(c["d_model"], c.get("norm_eps", 1e-5))

    def forward(self, ids, key_keep=None, causal=False):
        x = self.tok_emb(ids)
        for b in self.blocks: x = b(x, key_keep, causal)
        return self.final_norm(x)


class GRUBackbone(nn.Module):               # Model 3: the bidirectional GRU encoder of the A-2 seq2seq model
    kind = "gru"
    def __init__(self, c):
        super().__init__()
        self.c, self.hidden_size = c, 2 * c["hidden"]
        self.embedding = nn.Embedding(c["vocab_size"], c["emb_dim"])
        self.enc_rnn = nn.GRU(c["emb_dim"], c["hidden"], num_layers=c["enc_layers"], batch_first=True, bidirectional=True)

    def forward(self, ids, key_keep=None, causal=False):
        self.enc_rnn.flatten_parameters()
        x = self.embedding(ids)
        if key_keep is None:
            return self.enc_rnn(x)[0]
        lengths = key_keep.sum(1).clamp(min=1).cpu()
        out, _ = self.enc_rnn(pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False))
        return pad_packed_sequence(out, batch_first=True, total_length=ids.size(1))[0]

In [ ]:
# ---- finding and loading the checkpoints ----
def _ckpt_kind(folder):
    names = os.listdir(folder)
    if "a3_config.json" in names: return "a3"
    if "modern_config.json" in names: return "modern"
    if "gru_config.json" in names: return "gru"
    st = [n for n in names if n.endswith(".safetensors")]
    if st:
        from safetensors import safe_open
        with safe_open(os.path.join(folder, st[0]), "pt") as f:
            keys = list(f.keys())
        if any("enc_rnn" in k for k in keys): return "gru"
        if any("attn_norm" in k for k in keys): return "modern"
        if any(".ln1." in k for k in keys): return "a3"
    return None

WANT_KIND = {"m2": "a3", "m3": "gru", "m4": "modern", "m5": "modern"}

def find_checkpoint(key):
    p = MODEL_DIRS.get(key)
    if p:
        folder = p if os.path.isdir(p) else os.path.dirname(p)
        if _ckpt_kind(folder) is None:          # a dataset root: look one level deeper
            subs = [os.path.dirname(f) for f in glob.glob(os.path.join(folder, "**", "*"), recursive=True)
                    if f.endswith(("_config.json", ".safetensors"))]
            subs = [s for s in dict.fromkeys(subs) if "checkpoint-" not in s and _ckpt_kind(s)]
            assert subs, f"no checkpoint found under {p}"
            folder = subs[0]
        return folder
    cands = []
    for f in glob.glob("/kaggle/input/**/*", recursive=True):
        if f.endswith(("_config.json", ".safetensors")):
            folder = os.path.dirname(f)
            if "checkpoint-" in folder: continue
            if _ckpt_kind(folder) == WANT_KIND[key]: cands.append(folder)
    cands = list(dict.fromkeys(cands))
    hit = [c for c in cands if any(w in c.lower() for w in DISCOVER_KEYWORDS[key])]
    shared = sum(k == WANT_KIND[key] for k in WANT_KIND.values()) > 1   # m4/m5: same format, never guess
    assert hit or (len(cands) == 1 and not shared), \
        f"{key}: cannot pick a checkpoint automatically from {cands}; set MODEL_DIRS['{key}']"
    return (hit or cands)[0]

def _strip(sd):
    out = {}
    for k, v in sd.items():
        for pre in ("module.", "lm.", "model."):
            if k.startswith(pre): k = k[len(pre):]
        out[k] = v
    return out

def load_backbone(key, n_layers=None):
    """Phase 1 checkpoint -> (backbone, info). n_layers < full depth keeps only the lowest layers (KD student)."""
    folder = find_checkpoint(key)
    kind = _ckpt_kind(folder)
    names = os.listdir(folder)
    pt = [n for n in names if n.endswith("_model.pt")]
    if pt:
        sd = torch.load(os.path.join(folder, pt[0]), map_location="cpu", weights_only=True)
    else:
        from safetensors.torch import load_file
        sd = load_file(os.path.join(folder, [n for n in names if n.endswith(".safetensors")][0]))
    sd = _strip(sd)
    cfg_name = {"a3": "a3_config.json", "modern": "modern_config.json", "gru": "gru_config.json"}[kind]
    if cfg_name in names:
        c = json.load(open(os.path.join(folder, cfg_name)))
    elif kind == "gru":                          # Trainer checkpoint without config: read the sizes off the weights
        c = dict(vocab_size=sd["embedding.weight"].shape[0], emb_dim=sd["embedding.weight"].shape[1],
                 hidden=sd["enc_rnn.weight_hh_l0"].shape[1],
                 enc_layers=len([k for k in sd if re.fullmatch(r"enc_rnn\.weight_ih_l\d+", k)]))
    else:
        raise FileNotFoundError(f"{folder}: {cfg_name} is needed to know the number of heads")
    full_layers = c["enc_layers"] if kind == "gru" else c["n_layers"]
    if n_layers:
        assert n_layers <= full_layers
        if kind == "gru":
            c = {**c, "enc_layers": n_layers}
            sd = {k: v for k, v in sd.items() if not re.search(r"_l(\d+)", k) or int(re.search(r"_l(\d+)", k).group(1)) < n_layers}
        else:
            c = {**c, "n_layers": n_layers}
            sd = {k: v for k, v in sd.items() if not k.startswith("blocks.") or int(k.split(".")[1]) < n_layers}
    model = {"a3": A3Backbone, "modern": ModernBackbone, "gru": GRUBackbone}[kind](c)
    own = model.state_dict()
    used = {k: v for k, v in sd.items() if k in own}
    missing = [k for k in own if k not in used]
    assert not missing, f"weights missing from the checkpoint: {missing[:5]}"
    for k, v in used.items():
        assert v.shape == own[k].shape, f"{k}: checkpoint {tuple(v.shape)} vs model {tuple(own[k].shape)}"
    model.load_state_dict(used)
    tok = [os.path.join(folder, n) for n in names if n.startswith("bpe_tokenizer") and n.endswith(".json")]
    for where in (os.path.join(os.path.dirname(folder), "**"), "/kaggle/input/**"):     # next to the weights, then anywhere
        tok = tok or sorted(glob.glob(os.path.join(where, "bpe_tokenizer*.json"), recursive=True))
    assert tok, f"no bpe_tokenizer*.json found for {key}: add the Phase 1 tokenizer as an input"
    info = dict(folder=folder, kind=kind, config=c, full_layers=full_layers, n_layers=n_layers or full_layers,
                unused_keys=len(sd) - len(used), tokenizer=tok[0], prefix_lm=c.get("prefix_lm"))
    return model, info

In [ ]:
# ---- adaptation methods ----
class LoRALinear(nn.Module):
    """LoRA (Hu et al. 2021): h = W0 x + (alpha / r) B A x, A (r x in) random, B (out x r) = 0."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        return self.base(x) + (x @ self.A.t().to(x.dtype)) @ self.B.t().to(x.dtype) * self.scale


class LoRMALinear(nn.Module):
    """LoRMA+ (Bihany, Patel & Modi 2025): multiplicative update h = (I + (alpha / r) B A) W0 x, with A (r x out) random,
    B (out x r) = 0 so training starts at W0. Computed as z = W0 x, then z + (alpha / r) B (A z): never forms the d x d matrix."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        z = F.linear(x, self.base.weight)
        z = z + (z @ self.A.t().to(z.dtype)) @ self.B.t().to(z.dtype) * self.scale
        return z if self.base.bias is None else z + self.base.bias



class RSLoRALinear(LoRALinear):
    """rsLoRA (Kalajdzievski 2023): LoRA with the update scaled by alpha / sqrt(r) instead of alpha / r, so the update
    does not shrink as the rank grows."""
    def __init__(self, base, r, alpha):
        super().__init__(base, r, alpha)
        self.scale = alpha / math.sqrt(r)


def adapter_target_names(backbone, targets):
    names = []
    for n, m in backbone.named_modules():
        if not isinstance(m, nn.Linear): continue
        is_attn = ".attn." in n
        is_mlp = ".ffn." in n or ".mlp." in n
        if (is_attn and "attn" in targets) or (is_mlp and "mlp" in targets): names.append(n)
    return names

def inject(backbone, cls, targets, r, alpha):
    names = adapter_target_names(backbone, targets)
    for n in names:
        parent, child = backbone.get_submodule(n.rsplit(".", 1)[0]), n.rsplit(".", 1)[1]
        setattr(parent, child, cls(getattr(parent, child), r, alpha))
    return names

In [ ]:
# ---- sentences -> token ids (Phase 1 tokenizer), scoring (= evaluate_phase3.py) ----
_TOKENIZERS, _ENCODINGS = {}, {}
def get_encodings(tok_path):
    """Every sentence of every document as token ids, cached per tokenizer file content."""
    h = hashlib.md5(open(tok_path, "rb").read()).hexdigest()
    if h not in _ENCODINGS:
        t0 = time.time(); bpe = OffsetBPE(tok_path)
        enc = {d: [bpe.encode(s)[0] if s.strip() else np.array([bpe.eow_id], np.int64) for s in x["sents"]]
               for d, x in DOCS.items()}
        _TOKENIZERS[h], _ENCODINGS[h] = bpe, enc
        n = np.array([len(t) for d in TRAIN_IDS for t in enc[d]])
        log(f"tokenizer {tok_path} (md5 {h[:8]}, vocab {bpe.vocab_size}): {sum(len(v) for v in enc.values())} sentences "
            f"in {time.time() - t0:.0f}s; train tokens/sentence median {int(np.median(n))}, p90 {int(np.percentile(n, 90))}, "
            f"p99 {int(np.percentile(n, 99))}, max {n.max()}")
    return _TOKENIZERS[h], _ENCODINGS[h]

def score(pred, ids):
    """pred: doc id -> list of labels. Pooled over all sentences of `ids`, macro over the 13 official labels,
    zero_division=0 - the same numbers as evaluate_phase3.py."""
    from sklearn.metrics import precision_recall_fscore_support
    y = [l for d in ids for l in DOCS[d]["labels"]]
    p = [l for d in ids for l in pred[d]]
    P, R, F1, S = precision_recall_fscore_support(y, p, labels=LABELS, average=None, zero_division=0)
    return dict(macro_f1=float(F1.mean()), macro_p=float(P.mean()), macro_r=float(R.mean()),
                accuracy=float(np.mean([a == b for a, b in zip(y, p)])),
                per_label={l: dict(p=float(P[k]), r=float(R[k]), f1=float(F1[k]), support=int(S[k])) for k, l in enumerate(LABELS)})

def confusions(pred, ids, top=12):
    c = Counter((g, p) for d in ids for g, p in zip(DOCS[d]["labels"], pred[d]) if g != p)
    return c.most_common(top)

In [ ]:
# ---- windows of sentences, batches ----
def doc_windows(lens, W, overlap):
    """Consecutive sentences [i, j) with at most W tokens in total. overlap: the next window starts at the first
    sentence past the middle of the previous one, so (away from the document edges) every sentence is in 2 windows."""
    out, i, n = [], 0, len(lens)
    while i < n:
        j, tot = i, 0
        while j < n and tot + lens[j] <= W: tot += lens[j]; j += 1
        j = max(j, i + 1)
        out.append((i, j))
        if j >= n: break
        if overlap:
            k, acc = i, 0
            while k < j and acc + lens[k] <= tot / 2: acc += lens[k]; k += 1
            i = max(k, i + 1)
        else:
            i = j
    return out

def make_items(enc, ids, cfg):
    if cfg["context"] == "sentence":
        return [(d, i, i + 1) for d in ids for i in range(len(enc[d]))]
    return [(d, i, j) for d in ids for i, j in doc_windows([len(t) for t in enc[d]], cfg["window"], cfg["overlap"])]

def item_len(enc, it):
    d, i, j = it
    return sum(len(t) for t in enc[d][i:j])

def batches(items, enc, budget):
    """Consecutive groups of items with (number of items) x (longest item) <= budget tokens."""
    out, cur, mx = [], [], 0
    for it in items:
        L = item_len(enc, it)
        if cur and (len(cur) + 1) * max(mx, L) > budget:
            out.append(cur); cur, mx = [], 0
        cur.append(it); mx = max(mx, L)
    if cur: out.append(cur)
    return out

def make_batch(items, enc, teacher=None):
    """-> ids (B, T), key_keep (None if no padding), sent_idx (B, T) (-1 = padding), labels (B, S) (-100 = none),
    teacher logits (B, S, N_LABELS) or None, S."""
    B = len(items); T = max(item_len(enc, it) for it in items); S = max(j - i for _, i, j in items)
    ids = torch.zeros(B, T, dtype=torch.long); keep = torch.zeros(B, T, dtype=torch.bool)
    sid = torch.full((B, T), -1, dtype=torch.long); y = torch.full((B, S), -100, dtype=torch.long)
    tl = torch.zeros(B, S, N_LABELS) if teacher is not None else None
    for b, (d, i, j) in enumerate(items):
        pos = 0
        for k, t in enumerate(enc[d][i:j]):
            ids[b, pos:pos + len(t)] = torch.from_numpy(t); sid[b, pos:pos + len(t)] = k; pos += len(t)
        keep[b, :pos] = True
        if DOCS[d]["labels"] is not None:
            y[b, :j - i] = torch.tensor([L2I[l] for l in DOCS[d]["labels"][i:j]])
        if teacher is not None: tl[b, :j - i] = teacher[d][i:j].float()
    return ids, (None if bool(keep.all()) else keep), sid, y, tl, S

In [ ]:
# ---- sentence head (+ optional CRF over the sentence sequence) ----
class SentCRF(nn.Module):
    """Linear-chain CRF over the sentences of a window: learned role-to-role transition scores (no constraints).
    Sentences are a prefix of each row; padding (-100) only at the end."""
    def __init__(self, n):
        super().__init__()
        self.trans = nn.Parameter(torch.zeros(n, n))           # trans[i, j]: score of role j right after role i
        self.start = nn.Parameter(torch.zeros(n)); self.end = nn.Parameter(torch.zeros(n))

    def scores(self):
        return self.trans, self.start, self.end

    def nll(self, em, y):
        """em (B, S, n) float32, y (B, S). Summed NLL / number of sentences (same scale as the cross-entropy)."""
        B, S, n = em.shape
        mask = y != -100; yc = y.clamp(min=0); length = mask.sum(1)
        gold = self.start[yc[:, 0]] + em[:, 0].gather(1, yc[:, :1]).squeeze(1)
        alpha = self.start + em[:, 0]
        for t in range(1, S):
            m = mask[:, t]
            step = self.trans[yc[:, t - 1], yc[:, t]] + em[:, t].gather(1, yc[:, t:t + 1]).squeeze(1)
            gold = gold + torch.where(m, step, torch.zeros_like(step))
            alpha = torch.where(m[:, None], torch.logsumexp(alpha[:, :, None] + self.trans, 1) + em[:, t], alpha)
        gold = gold + self.end[yc.gather(1, (length - 1).clamp(min=0)[:, None]).squeeze(1)]
        log_z = torch.logsumexp(alpha + self.end, -1)
        has = length > 0
        return (log_z - gold)[has].sum() / mask.sum().clamp(min=1)


class SentenceTagger(nn.Module):
    def __init__(self, backbone, n_labels, dropout, causal, head="linear"):
        super().__init__()
        self.backbone, self.causal = backbone, causal
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(backbone.hidden_size, n_labels)
        self.crf = SentCRF(n_labels) if head == "crf" else None

    def forward(self, ids, key_keep, sent_idx, S):
        """-> (B, S, n_labels) float32: the mean of each sentence's token vectors, through dropout + linear."""
        h = self.backbone(ids, key_keep, self.causal).float()
        B, T, d = h.shape
        valid = sent_idx >= 0; idx = sent_idx.clamp(min=0)
        sums = h.new_zeros(B, S, d).scatter_add_(1, idx[..., None].expand(-1, -1, d), h * valid[..., None])
        cnt = h.new_zeros(B, S).scatter_add_(1, idx, valid.float())
        return self.head(self.drop(sums / cnt.clamp(min=1)[..., None])).float()

    def decoder(self):
        """None = per-sentence argmax; else (trans, start, end) numpy arrays for viterbi()."""
        if self.crf is None: return None
        return tuple(x.detach().float().cpu().numpy() for x in self.crf.scores())


def setup_method(model, cfg):
    """Freezes / wraps parameters for the run's method. Returns a list of (param group name, params, lr)."""
    bb, method = model.backbone, cfg["method"]
    for p in bb.parameters(): p.requires_grad = method in ("full", "kd", "partial")
    adapters = []
    if method == "partial":                          # train the top layers only
        (bb.embedding if bb.kind == "gru" else bb.tok_emb).requires_grad_(False)
        if bb.kind == "gru":
            L = bb.c["enc_layers"]; keep_from = L - max(1, round(L * cfg["partial_trainable_frac"]))
            for n, p in bb.enc_rnn.named_parameters():
                if int(re.search(r"_l(\d+)", n).group(1)) < keep_from: p.requires_grad = False
        else:
            L = len(bb.blocks); keep_from = L - max(1, round(L * cfg["partial_trainable_frac"]))
            for b in bb.blocks[:keep_from]: b.requires_grad_(False)
        log(f"partial: layers {keep_from}..{L - 1} trainable")
    elif method in ("lora", "rslora", "lorma"):
        assert bb.kind != "gru", f"{method} is implemented for Linear layers; the GRU encoder has none"
        cls = {"lora": LoRALinear, "rslora": RSLoRALinear, "lorma": LoRMALinear}[method]
        names = inject(bb, cls, cfg["adapter_targets"], cfg["rank"], cfg["alpha"])
        adapters = [p for n, p in bb.named_parameters() if n.endswith((".A", ".B"))]
        log(f"{method}: rank {cfg['rank']}, alpha {cfg['alpha']}, {len(names)} adapted Linear layers, e.g. {names[:4]}")
    elif method == "bitfit":
        adapters = [p for n, p in bb.named_parameters() if n.endswith("bias") or re.search(r"bias_(ih|hh)", n)]
        assert adapters, "this backbone has no bias terms: BitFit is not applicable (Models 4/5 use bias-free layers)"
        log(f"bitfit: {len(adapters)} bias tensors trainable")
    else:
        assert method in ("full", "kd"), f"unknown method {method}"
    for p in adapters: p.requires_grad = True
    ad_ids = {id(p) for p in adapters}
    backbone_params = [p for p in bb.parameters() if p.requires_grad and id(p) not in ad_ids]
    head = list(model.head.parameters()) + (list(model.crf.parameters()) if model.crf is not None else [])
    return [("head", head, cfg["lr_head"]),
            ("adapter", adapters, cfg["lr_adapter"]),
            ("backbone", backbone_params, cfg["lr_backbone"])]

In [ ]:
# ---- prediction ----
@torch.no_grad()
def doc_logits(model, enc, ids, cfg):
    """Per-sentence logits (n_sentences, N_LABELS) per document, averaged over the windows containing the sentence."""
    model.eval()
    acc = {d: torch.zeros(len(enc[d]), N_LABELS) for d in ids}
    cnt = {d: torch.zeros(len(enc[d]), 1) for d in ids}
    items = sorted(make_items(enc, ids, cfg), key=lambda it: item_len(enc, it))     # similar lengths: less padding
    for chunk in batches(items, enc, 2 * cfg["batch_tokens"]):
        x, keep, sid, _, _, S = make_batch(chunk, enc)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
            lg = model(x.to(DEVICE), None if keep is None else keep.to(DEVICE), sid.to(DEVICE), S).float().cpu()
        for b, (d, i, j) in enumerate(chunk):
            acc[d][i:j] += lg[b, :j - i]; cnt[d][i:j] += 1
    return {d: acc[d] / cnt[d].clamp(min=1) for d in ids}

def viterbi(em, trans, start, end):
    """Highest-scoring label path. em (n, L) numpy; trans[i, j] = score of j after i."""
    n, L = em.shape
    bp = np.zeros((n, L), np.int64)
    sc = start + em[0]
    for t in range(1, n):
        m = sc[:, None] + trans
        bp[t] = m.argmax(0); sc = m.max(0) + em[t]
    path = np.zeros(n, np.int64); path[-1] = (sc + end).argmax()
    for t in range(n - 1, 0, -1): path[t - 1] = bp[t, path[t]]
    return path

def predict(logits, dec):
    """doc id -> list of labels (argmax per sentence, or Viterbi over the whole document with the CRF transitions)."""
    out = {}
    for d, lg in logits.items():
        k = lg.argmax(-1).numpy() if dec is None else viterbi(lg.numpy(), *dec)
        out[d] = [LABELS[i] for i in k]
    return out

def log_report(vm, vpred, ids):
    for dom in ("CL", "IT"):
        dm = score(vpred, [d for d in ids if DOCS[d]["domain"] == dom])
        log(f"  {dom}: macro-F1 {dm['macro_f1']:.4f} (P {dm['macro_p']:.4f} R {dm['macro_r']:.4f}), accuracy {dm['accuracy']:.4f}")
    log(f"  {'label':24s} {'P':>6s} {'R':>6s} {'F1':>6s} {'support':>7s}")
    for l in LABELS:
        v = vm["per_label"][l]
        log(f"  {l:24s} {v['p']:6.3f} {v['r']:6.3f} {v['f1']:6.3f} {v['support']:7d}")
    log("  most frequent confusions (gold -> predicted, count): "
        + ", ".join(f"{g}->{p} {c}" for (g, p), c in confusions(vpred, ids)))

In [ ]:
# ---- one experiment ----
RESULTS, TEACHER_LOGITS = [], {}

def count_params(m, trainable=False):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable))

def run_experiment(run):
    cfg = {**COMMON, **run}
    if SMOKE: cfg.update(epochs=1, max_train_windows=16)
    unset = [k for k, v in cfg.items() if v is ...]
    assert not unset, f"run {cfg['name']}: replace the `...` placeholder with a value for {unset}"
    name = cfg["name"]
    out_dir = os.path.join(WORK, name); os.makedirs(out_dir, exist_ok=True)
    section(f"RUN {name}: model {cfg['model']}, method {cfg['method']}, context {cfg['context']}, head {cfg['head']}")
    log("config:", json.dumps({k: v for k, v in cfg.items() if k not in COMMON or COMMON[k] != v}))
    assert cfg["context"] in ("doc", "sentence") and cfg["head"] in ("linear", "crf")
    set_seed(cfg["seed"])
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()

    backbone, info = load_backbone(cfg["model"], cfg["student_layers"])
    log(f"checkpoint {info['folder']} (kind {info['kind']}, layers {info['n_layers']}/{info['full_layers']}, "
        f"prefix_lm={info['prefix_lm']}, unused checkpoint tensors {info['unused_keys']})")
    if cfg["model"] in ("m4", "m5"):                     # same file format: make sure it is the right checkpoint
        assert bool(info["prefix_lm"]) == (cfg["model"] == "m5"), (
            f"{cfg['model']} loaded a checkpoint with prefix_lm={info['prefix_lm']} - wrong MODEL_DIRS['{cfg['model']}']?")
    bpe, enc_full = get_encodings(info["tokenizer"])
    limit = cfg["max_sent_tokens"]
    if info["kind"] != "gru":
        assert cfg["window"] <= info["config"]["max_seq_len"], "window longer than the model's context"
        limit = min(limit, cfg["window"] if cfg["context"] == "doc" else info["config"]["max_seq_len"])
    enc = {d: [t[:limit] for t in v] for d, v in enc_full.items()}
    n_cut = sum(len(t) > limit for d in TRAIN_IDS for t in enc_full[d])
    log(f"sentences cut to {limit} tokens: {n_cut} of {sum(len(enc_full[d]) for d in TRAIN_IDS)} training sentences")

    model = SentenceTagger(backbone, N_LABELS, cfg["head_dropout"],
                           causal=(cfg["attn"] == "causal" and info["kind"] != "gru"), head=cfg["head"])
    train_ids = TRAIN_IDS + (VAL_IDS if cfg["train_on_val"] else [])
    if cfg["train_on_val"]:
        log(f"train_on_val: training on {len(train_ids)} docs, keeping the last epoch (dev scores below are on TRAINING docs)")
    if cfg["head"] == "crf" and cfg["class_weighting"] != "none":
        log("note: class_weighting is not used by the CRF loss")
    groups = setup_method(model, cfg)
    model.to(DEVICE)
    n_total, n_train = count_params(model), count_params(model, True)
    log(f"parameters: total {n_total / 1e6:.2f}M, trainable {n_train / 1e6:.3f}M ({100 * n_train / n_total:.2f}%)")

    teacher = None
    if cfg["method"] == "kd":
        t = cfg["teacher"]
        if t not in TEACHER_LOGITS:
            hits = glob.glob(f"/kaggle/input/**/{t}/teacher_logits.pt", recursive=True)
            assert hits, f"teacher run {t} has not run in this session and no {t}/teacher_logits.pt input was found"
            TEACHER_LOGITS[t] = torch.load(hits[0])
        teacher = TEACHER_LOGITS[t]
        assert all(d in teacher and len(teacher[d]) == len(enc[d]) for d in train_ids), "teacher logits do not cover the training docs"
        log(f"kd: teacher {t}, temperature {cfg['kd_temperature']}, alpha {cfg['kd_alpha']}")

    train_items = make_items(enc, train_ids, cfg)
    per_epoch = min(len(train_items), cfg["max_train_windows"] or len(train_items))
    class_w = None
    if cfg["class_weighting"] == "sqrt_inv":
        cnt = Counter(l for d in train_ids for l in DOCS[d]["labels"])
        top = max(cnt.values())
        class_w = torch.tensor([min(10.0, math.sqrt(top / max(1, cnt[l]))) for l in LABELS], device=DEVICE)
        log("class weights:", dict(zip(LABELS, [round(x, 2) for x in class_w.tolist()])))
    n_batches = len(batches(train_items[:per_epoch], enc, cfg["batch_tokens"]))
    steps = math.ceil(n_batches / cfg["grad_accum"]) * cfg["epochs"]
    pg = [dict(params=[p for p in ps if p.ndim >= 2], lr=lr, weight_decay=cfg["weight_decay"]) for _, ps, lr in groups] + \
         [dict(params=[p for p in ps if p.ndim < 2], lr=lr, weight_decay=0.0) for _, ps, lr in groups]
    opt = torch.optim.AdamW([g for g in pg if g["params"]])
    warm = max(1, int(cfg["warmup_frac"] * steps))
    lr_at = lambda s: (s + 1) / warm if s < warm else max(0.0, (steps - s) / max(1, steps - warm))   # warmup, linear decay
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda" and AMP_DTYPE == torch.float16))
    log(f"train items/epoch {per_epoch} ({'windows of <= ' + str(cfg['window']) + ' tokens' if cfg['context'] == 'doc' else 'single sentences'}), "
        f"~{n_batches} batches of <= {cfg['batch_tokens']} tokens, optimizer steps ~{steps}, warmup {warm}")

    best, best_state, history, bad = -1.0, None, [], 0
    rng = random.Random(cfg["seed"])                     # own generator: parallel runs do not disturb each other
    t_train = time.time()
    for ep in range(cfg["epochs"]):
        model.train(); t0 = time.time()
        rng.shuffle(train_items)
        bl = batches(train_items[:per_epoch], enc, cfg["batch_tokens"])
        tot, nb = 0.0, 0
        opt.zero_grad(set_to_none=True)
        for bi, chunk in enumerate(bl):
            x, keep, sid, y, tl, S = make_batch(chunk, enc, teacher)
            x, sid, y = x.to(DEVICE), sid.to(DEVICE), y.to(DEVICE)
            with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
                lg = model(x, None if keep is None else keep.to(DEVICE), sid, S)
            if model.crf is not None: loss = model.crf.nll(lg, y)
            else: loss = F.cross_entropy(lg.view(-1, N_LABELS), y.view(-1), weight=class_w, ignore_index=-100)
            if teacher is not None:
                m = y.view(-1) != -100
                T_ = cfg["kd_temperature"]
                kd = F.kl_div(F.log_softmax(lg.view(-1, N_LABELS)[m] / T_, -1),
                              F.softmax(tl.to(DEVICE).view(-1, N_LABELS)[m] / T_, -1), reduction="batchmean") * T_ * T_
                loss = (1 - cfg["kd_alpha"]) * loss + cfg["kd_alpha"] * kd
            scaler.scale(loss / cfg["grad_accum"]).backward()
            nb += 1; tot += loss.item()
            if nb % cfg["grad_accum"] == 0 or bi == len(bl) - 1:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], cfg["grad_clip"])
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        vm = score(predict(doc_logits(model, enc, VAL_IDS, cfg), model.decoder()), VAL_IDS)
        history.append(dict(epoch=ep + 1, train_loss=tot / max(1, nb), val_macro_f1=vm["macro_f1"], val_p=vm["macro_p"],
                            val_r=vm["macro_r"], val_acc=vm["accuracy"], seconds=time.time() - t0))
        log(f"  epoch {ep + 1:2d}/{cfg['epochs']}: train loss {tot / max(1, nb):.4f} | dev macro-F1 {vm['macro_f1']:.4f} "
            f"(P {vm['macro_p']:.4f} R {vm['macro_r']:.4f}, acc {vm['accuracy']:.4f}) | {time.time() - t0:.0f}s")
        if cfg["train_on_val"] or vm["macro_f1"] > best:     # train_on_val: always the latest epoch
            best, bad = vm["macro_f1"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}
        else:
            bad += 1
            if cfg["patience"] and bad >= cfg["patience"]:
                log("  early stop"); break
    train_secs = time.time() - t_train
    model.load_state_dict(best_state, strict=False)
    best_ep = history[-1]["epoch"] if cfg["train_on_val"] else max(history, key=lambda h: h["val_macro_f1"])["epoch"]

    # dev report at the best epoch
    dec = model.decoder()
    vl = doc_logits(model, enc, VAL_IDS, cfg)
    vpred = predict(vl, dec)
    vm = score(vpred, VAL_IDS)
    log(f"best epoch {best_ep}: dev macro-F1 {vm['macro_f1']:.4f}, P {vm['macro_p']:.4f}, R {vm['macro_r']:.4f}, "
        f"accuracy {vm['accuracy']:.4f} (decode {'viterbi' if dec else 'argmax'})")
    log_report(vm, vpred, VAL_IDS)

    # test predictions
    t0 = time.time()
    tl_ = doc_logits(model, enc, TEST_IDS, cfg)
    infer_secs = time.time() - t0
    tpred = predict(tl_, dec)
    for d in TEST_IDS:
        assert len(tpred[d]) == len(DOCS[d]["sents"]) and all(l in L2I for l in tpred[d]), d
    with open(os.path.join(out_dir, "prediction.json"), "w", encoding="utf-8") as f:
        json.dump({d: tpred[d] for d in TEST_IDS}, f, ensure_ascii=False)
    with zipfile.ZipFile(os.path.join(out_dir, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(out_dir, "prediction.json"), "prediction.json")
    tc = Counter(l for d in TEST_IDS for l in tpred[d])
    log(f"test: {sum(tc.values())} sentences in {len(TEST_IDS)} docs ({', '.join(f'{l}={tc[l]}' for l in LABELS)}), "
        f"inference {infer_secs:.0f}s")
    torch.save(dict(dev=vl, test=tl_, decoder=dec), os.path.join(out_dir, "logits.pt"))   # for later ensembles

    # teacher logits (train + dev docs), saved for every run: any run can be a KD teacher later, in this session or
    # in a later one (add the bundle as an input; the KD run finds <teacher>/teacher_logits.pt under /kaggle/input).
    # Skipped in SMOKE mode unless a KD run of this session needs them.
    if not SMOKE or any(r.get("teacher") == name for r in RUNS):
        TEACHER_LOGITS[name] = {d: v.half() for d, v in doc_logits(model, enc, TRAIN_IDS + VAL_IDS, cfg).items()}
        torch.save(TEACHER_LOGITS[name], os.path.join(out_dir, "teacher_logits.pt"))
        log("saved teacher logits (usable for a later KD run)")

    peak = torch.cuda.max_memory_allocated() / 2**30 if DEVICE.type == "cuda" else 0.0
    dom_f1 = {dom: score(vpred, [d for d in VAL_IDS if DOCS[d]["domain"] == dom])["macro_f1"] for dom in ("CL", "IT")}
    res = dict(name=name, model=cfg["model"], method=cfg["method"], context=cfg["context"], head=cfg["head"],
               attn=cfg["attn"] if info["kind"] != "gru" else "bigru",
               layers=f"{info['n_layers']}/{info['full_layers']}", total_params=n_total, trainable_params=n_train,
               best_epoch=best_ep, val_macro_f1=vm["macro_f1"], val_macro_p=vm["macro_p"], val_macro_r=vm["macro_r"],
               val_accuracy=vm["accuracy"], val_cl_f1=dom_f1["CL"], val_it_f1=dom_f1["IT"],
               val_per_label={l: v["f1"] for l, v in vm["per_label"].items()},
               val_confusions=[[g, p, c] for (g, p), c in confusions(vpred, VAL_IDS, 30)],
               train_minutes=train_secs / 60, test_inference_seconds=infer_secs, peak_gpu_gb=peak, history=history,
               config=cfg, checkpoint=info["folder"])
    json.dump(res, open(os.path.join(out_dir, "val_metrics.json"), "w"), indent=1)
    json.dump({d: vpred[d] for d in VAL_IDS}, open(os.path.join(out_dir, "val_predictions.json"), "w"))
    if cfg["save_weights"]:
        torch.save(dict(trainable=best_state, config=cfg, backbone_config=info["config"]), os.path.join(out_dir, "trainable_weights.pt"))
    log(f"train {train_secs / 60:.1f} min, peak GPU memory {peak:.2f} GB, outputs in {out_dir}")
    RESULTS.append(res)
    del model, opt, backbone; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return res

In [ ]:
# ---- run everything: one worker per GPU, each takes the next run from the queue (2 x T4 -> 2 runs at a time) ----
# torch.cuda.set_device is per thread, and DEVICE = "cuda" means "this thread's GPU", so every run stays on its GPU.
# A KD run whose teacher is trained in this same session waits until the teacher has finished.
SELECTED = [r for r in RUNS if not RUN_ONLY or r["name"] in RUN_ONLY]
GPU_IDS = json.loads(os.environ.get("P3_GPU_IDS", "null")) or (list(range(torch.cuda.device_count())) if DEVICE.type == "cuda" else [None])
GPU_IDS = GPU_IDS[:max(1, len(SELECTED))]
_queue, _qlock, _finished = list(SELECTED), threading.Lock(), set()
_in_session = {r["name"] for r in SELECTED}

def _worker(gpu):
    if gpu is not None: torch.cuda.set_device(gpu)
    while True:
        with _qlock:
            if not _queue: return
            ready = [r for r in _queue if r.get("teacher") not in _in_session or r["teacher"] in _finished]
            run = ready[0] if ready else None
            if run: _queue.remove(run)
        if run is None:
            time.sleep(20); continue
        log(f"starting {run['name']}")
        try:
            run_experiment(run)
        except Exception as e:                           # one failing run must not lose the others
            import traceback
            log(f"RUN {run['name']} FAILED: {e!r}\n{traceback.format_exc()}")
            gc.collect()
            if DEVICE.type == "cuda": torch.cuda.empty_cache()
        finally:
            with _qlock: _finished.add(run["name"])

section(f"RUNNING {[r['name'] for r in SELECTED]} on {len(GPU_IDS)} worker(s): GPUs {GPU_IDS}")
_threads = [threading.Thread(target=_worker, args=(g,), name=f"gpu{g if g is not None else 'cpu'}-{i}") for i, g in enumerate(GPU_IDS)]
for t in _threads: t.start()
for t in _threads: t.join()

In [ ]:
# ---- summary + one zip to download ----
section("SUMMARY")
log(f"{'run':18s} {'model':5s} {'method':7s} {'context':8s} {'head':6s} {'layers':6s} {'total':>8s} {'trainable':>10s} "
    f"{'best ep':>7s} {'dev F1':>7s} {'CL F1':>6s} {'IT F1':>6s} {'acc':>6s} {'train min':>9s} {'infer s':>7s} {'peak GB':>7s}")
for r in RESULTS:
    log(f"{r['name']:18s} {r['model']:5s} {r['method']:7s} {r['context']:8s} {r['head']:6s} {r['layers']:6s} "
        f"{r['total_params'] / 1e6:7.1f}M {r['trainable_params'] / 1e6:9.3f}M {r['best_epoch']:7d} {r['val_macro_f1']:7.4f} "
        f"{r['val_cl_f1']:6.4f} {r['val_it_f1']:6.4f} {r['val_accuracy']:6.4f} {r['train_minutes']:9.1f} "
        f"{r['test_inference_seconds']:7.0f} {r['peak_gpu_gb']:7.2f}")
log("per-label dev F1:")
for r in RESULTS:
    log(f"  {r['name']:18s} " + " ".join(f"{l}={v:.2f}" for l, v in r["val_per_label"].items()))

json.dump(RESULTS, open(os.path.join(WORK, "phase3_results_summary.json"), "w"), indent=1)
_log.flush()
bundle = os.path.join(WORK, "phase3_results_bundle.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(os.path.join(WORK, "**", "*"), recursive=True):
        if (f.endswith((".txt", ".json", ".zip", "logits.pt"))          # teacher_logits.pt: reusable as a KD input
                and not f.endswith("phase3_results_bundle.zip")):
            z.write(f, os.path.relpath(f, WORK))
log(f"bundle: {bundle} ({os.path.getsize(bundle) / 1e6:.1f} MB)")